In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

sns.set_context("poster")
sns.set_style("ticks")

In [ ]:
fi = pd.read_parquet("0.parquet")
spearman = pd.read_parquet("1.parquet")
weights = pd.read_parquet("2.parquet")
weights["AbsWeight"] = weights.Weight.abs()
weights["AbsGTWeight"] = weights.GTWeight.abs()

In [ ]:
id_cols = [
    "trainer.model_builder.param",
    "trainer.dataset.noise_level",
    "trainer.dataset.seed",
]

# L2

## Main

In [ ]:
ax = sns.lineplot(
    weights[weights.Feature == "Feat. 1"],
    x="trainer.dataset.noise_level",
    y="L2",
    hue="trainer.model_builder.param",
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.get_legend().set_title(None)
ax.set_ylabel(r"$|| \hat{W} - W_{g.t.} ||_F$")
_ = ax.set_xlabel("Noise level")
plt.savefig("../../paper/figs/simulation/L2_ground_truth.pdf", bbox_inches="tight")
plt.show()

# Training duration

In [ ]:
ax = sns.lineplot(
    weights[weights.Feature == "Feat. 1"],
    x="trainer.dataset.noise_level",
    y="training_duration",
    hue="trainer.model_builder.param",
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.get_legend().set_title(None)
ax.set_ylabel("Training duration (s)")
_ = ax.set_xlabel("Noise level")
plt.savefig("../../paper/figs/simulation/training_duration.pdf", bbox_inches="tight")
plt.show()

## Diagonal vs off diagonal

In [ ]:
weights["Type"] = np.where(weights.Feature.str.contains("x"), "Interaction", "Feature")

In [ ]:
L2_per_type = (
    weights.groupby(id_cols + ["Type"])
    .apply(
        lambda x: (x.GTWeight - x.Weight).abs().mean(),
        include_groups=False,
    )
    .reset_index(name="L2")
)

In [ ]:
ax = sns.lineplot(
    L2_per_type.rename(columns={"trainer.model_builder.param": "Param"}),
    x="trainer.dataset.noise_level",
    y="L2",
    hue="Param",
    style="Type",
    marker="o",
    errorbar="sd",
)
ax.get_legend().set_title(None)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
ax.set_ylabel("Avg dist. to g.t.")
ax.set_xlabel("Noise level")

# Spearman

In [ ]:
ax = sns.lineplot(
    spearman[spearman.split == "test"],
    x="trainer.dataset.noise_level",
    y="mean",
    hue="trainer.model_builder.param",
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.get_legend().set_title(None)
ax.set_xlabel("Noise level")
ax.set_ylabel("Spearman")
plt.savefig("../../paper/figs/simulation/spearman.pdf", bbox_inches="tight")
plt.show()

# Weighted $\tau$

##  w.r.t. ground-truth

In [ ]:
weightedtau = (
    weights.groupby(
        id_cols,
    )
    .apply(
        lambda x: stats.weightedtau(x.GTWeight.abs(), x.Weight.abs()).statistic,
        include_groups=False,
    )
    .reset_index(name="Weighted $\\tau$")
)

In [ ]:
ax = sns.lineplot(
    weightedtau.rename(columns={"trainer.model_builder.param": "Param"}),
    x="trainer.dataset.noise_level",
    y="Weighted $\\tau$",
    hue="Param",
    style="Param",
    dashes=False,
    markers=True,
    errorbar="sd",
)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1), title=None)
ax.set_xlabel("Noise level")
ax.set_title("Weighted $\\tau$ to g.t. weights", pad=20)

##  FI/weights

In [ ]:
weightedtau = weights.merge(fi[fi.split == "test"], on=id_cols + ["Feature"])
weightedtau = (
    weightedtau.groupby(
        id_cols,
    )
    .apply(
        lambda x: stats.weightedtau(x.Weight.abs(), x["mean"].abs()).statistic,
        include_groups=False,
    )
    .reset_index(name="Weighted $\\tau$")
)

In [ ]:
ax = sns.lineplot(
    weightedtau,
    x="trainer.dataset.noise_level",
    y="Weighted $\\tau$",
    hue="trainer.model_builder.param",
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1), title=None)
ax.set_xlabel("Noise level")
ax.set_title("Weighted $\\tau$ between weights and FIs", pad=20)
plt.savefig("../../paper/figs/simulation/weighted_tau.pdf", bbox_inches="tight")
plt.show()